# Week 3: Risk Metrics and Portfolio Construction

Builds a reusable risk-metrics module (`src/risk.py`), applies it to the Week 2 factor portfolios, then compares portfolio construction methods.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd

sys.path.append("../../src")
from risk import compute_risk_metrics, infer_periods_per_year, max_drawdown
from factor_utils import performance_table

PORTFOLIOS = "../../data/portfolios"
REPORTS = "../../reports/week3"
os.makedirs(REPORTS, exist_ok=True)

## Task 1: Risk Metrics

`compute_risk_metrics()` in `src/risk.py` reports, for any return series:

| Metric | Definition | Basis |
| --- | --- | --- |
| Annual return (CAGR) | compound growth per year | annualized |
| Annual return (mean) | average periodic return × periods per year | annualized |
| Volatility | std of periodic returns × √(periods per year) | annualized |
| Downside deviation | √mean(min(r − rf, 0)²) × √(periods per year) | annualized |
| Sharpe | annual mean excess return ÷ volatility | annualized |
| Sortino | annual mean excess return ÷ downside deviation | annualized |
| Max drawdown | worst fall from a previous peak | over the full period |
| Calmar | CAGR ÷ \|max drawdown\| | annualized |
| Skewness, excess kurtosis | shape of the periodic return distribution | per period (not annualized) |
| Hit rate | share of periods with a positive return | per period |

**Annualization follows the data's frequency:** daily → 252, monthly → 12, inferred from the dates (the function refuses to guess if the spacing is unclear). Risk-free rate = 0 throughout: correct for the zero-cost long–short, and it keeps every row on the same basis.

### 1.1 Load the Week 2 portfolios
Monthly equal-weight quintile returns (Q1–Q5) and the long–short (Q5 − Q1) for value and momentum, July 2021 – August 2026.

In [ ]:
value_ret = pd.read_csv(f"{PORTFOLIOS}/value_quintile_returns.csv", index_col=0, parse_dates=True)
mom_ret = pd.read_csv(f"{PORTFOLIOS}/momentum_quintile_returns.csv", index_col=0, parse_dates=True)
COLS = ["Q1", "Q2", "Q3", "Q4", "Q5", "long_short"]

print("Frequency inferred from the dates:", infer_periods_per_year(value_ret.index), "periods per year (monthly)")
print("Months:", len(value_ret), "|", value_ret.index.min().date(), "to", value_ret.index.max().date())

### 1.2 Annualization check: daily vs monthly
The same portfolio (equal-weight across all 50 stocks) measured on **daily** and on **monthly** returns. If annualization is right, annual return, volatility, Sharpe and drawdown should come out nearly the same. Shape statistics (skewness, kurtosis) are *not* annualized, so they differ by frequency and must always state it.

In [ ]:
prices = pd.read_csv("../../data/raw/prices_2020.csv", parse_dates=["date"])
px = prices.pivot(index="date", columns="ticker", values="adj_close").loc["2021-07-01":"2026-08-31"]
ew_daily = px.pct_change().mean(axis=1).dropna().rename("Equal-weight, daily returns")
ew_monthly = (1 + ew_daily).resample("ME").prod().sub(1).rename("Equal-weight, monthly returns")

freq_check = pd.concat([compute_risk_metrics(ew_daily), compute_risk_metrics(ew_monthly)])
freq_check[["periods_per_year", "n_periods", "ann_return_cagr", "ann_volatility", "sharpe",
            "max_drawdown", "calmar", "skewness", "excess_kurtosis"]].round(3)

**What goes wrong with the wrong factor:** the same monthly returns annualized as if they were daily.

In [ ]:
wrong = compute_risk_metrics(ew_monthly, periods_per_year=252)
right = compute_risk_metrics(ew_monthly)
pd.DataFrame({
    "correct (12 per year)": right[["ann_volatility", "sharpe"]].iloc[0],
    "wrong (252 per year)": wrong[["ann_volatility", "sharpe"]].iloc[0],
}).round(2)

### 1.3 Risk summary table: Week 2 quintile portfolios and long–short factors
Monthly returns, annualized with 12. Saved to `reports/week3/risk_summary.csv`.

In [ ]:
risk_summary = pd.concat({
    "Value": compute_risk_metrics(value_ret[COLS]),
    "Momentum": compute_risk_metrics(mom_ret[COLS]),
})
risk_summary.to_csv(f"{REPORTS}/risk_summary.csv")

pct = ["ann_return_cagr", "ann_return_mean", "ann_volatility", "downside_deviation", "max_drawdown", "hit_rate"]
ratios = ["sharpe", "sortino", "calmar", "skewness", "excess_kurtosis"]
show = risk_summary[pct + ratios].copy()
show[pct] = (show[pct] * 100).round(1)
show[ratios] = show[ratios].round(2)
show.columns = ["CAGR %", "Mean ann. %", "Vol %", "Downside dev %", "Max DD %", "Hit rate %",
                "Sharpe", "Sortino", "Calmar", "Skew", "Excess kurt"]
show

### 1.4 Checks
1. **Consistency with Week 2:** Sharpe, volatility, max drawdown and hit rate must match Week 2's `performance_table` exactly (same data, same basis).
2. **Hand calculation:** Sortino and Calmar for the value long–short, recomputed from the raw monthly returns.

In [ ]:
w2 = pd.concat({"Value": performance_table(value_ret), "Momentum": performance_table(mom_ret)})
pairs = {"sharpe": "sharpe_ratio", "ann_volatility": "annualized_volatility", "max_drawdown": "max_drawdown", "hit_rate": "hit_rate"}
max_diff = max((risk_summary[a] - w2[b].astype(float)).abs().max() for a, b in pairs.items())
print(f"Largest difference vs Week 2: {max_diff:.1e}")

r = value_ret["long_short"]
downside = np.sqrt((np.minimum(r, 0) ** 2).mean()) * np.sqrt(12)
sortino_hand = r.mean() * 12 / downside
cagr_hand = (1 + r).prod() ** (12 / len(r)) - 1
calmar_hand = cagr_hand / abs(max_drawdown(r))
print(f"Value long-short Sortino: hand {sortino_hand:.4f} | code {risk_summary.loc[('Value', 'long_short'), 'sortino']:.4f}")
print(f"Value long-short Calmar:  hand {calmar_hand:.4f} | code {risk_summary.loc[('Value', 'long_short'), 'calmar']:.4f}")

### 1.5 Reconciling Sortino and Calmar with Sharpe and drawdown
Sortino and Calmar are read *next to* Sharpe and drawdown, not alone:
- **Sortino ÷ Sharpe = volatility ÷ downside deviation.** Above about 1.4 means losses are a smaller share of the swings than in a symmetric distribution, which usually goes with positive skew.
- **Calmar uses compound growth (CAGR) over the worst drawdown.** The gap between mean and CAGR is the "volatility drag": the more volatile the portfolio, the more the simple average overstates what $1 actually grew at.

In [ ]:
ls = risk_summary.xs("long_short", level=1)
recon = pd.DataFrame({
    "Sharpe": ls["sharpe"],
    "Sortino": ls["sortino"],
    "Sortino / Sharpe": ls["sortino"] / ls["sharpe"],
    "Vol %": ls["ann_volatility"] * 100,
    "Downside dev %": ls["downside_deviation"] * 100,
    "Skew": ls["skewness"],
    "Excess kurt": ls["excess_kurtosis"],
    "Mean ann. %": ls["ann_return_mean"] * 100,
    "CAGR %": ls["ann_return_cagr"] * 100,
    "Volatility drag (pp)": (ls["ann_return_mean"] - ls["ann_return_cagr"]) * 100,
    "Max DD %": ls["max_drawdown"] * 100,
    "Calmar": ls["calmar"],
}).round(2)
recon

### 1.6 Risk metrics: notes

- **Function:** `compute_risk_metrics()` in `src/risk.py` takes a return series or a table of them and returns volatility, Sharpe, Sortino, max drawdown, Calmar, skewness, excess kurtosis and hit rate (plus CAGR, mean return and downside deviation).
- **Consistent basis:** every return and risk figure is annualized with the same factor, taken from the data's frequency (12 for the Week 2 monthly portfolios). Skewness, kurtosis and hit rate are per-period and labeled as such. Risk-free rate = 0 for every row.
- **Annualization verified:** the same equal-weight portfolio measured daily and monthly gives nearly the same CAGR (24.5% vs 24.4%), volatility (19.6% vs 19.3%) and Sharpe (1.22 vs 1.24). Annualizing monthly data with 252 by mistake would show 88% volatility and a Sharpe of 5.7, exactly the error the quality check warns about.
- **Checks:** Sharpe, volatility, drawdown and hit rate match Week 2 exactly; Sortino and Calmar match hand calculations.
- **Value vs momentum (long–short):** value ranks ahead on every risk-adjusted measure: Sharpe 0.74 vs 0.48, Sortino 1.41 vs 0.78, Calmar 0.45 vs 0.28. Sortino and Calmar agree with Sharpe and drawdown, so the ranking is not an artifact of one metric.
- **Momentum's tails:** excess kurtosis of 2.9 (vs 0.9 for value) reflects its sudden crashes (−29% in Jan 2023, −30% in Jul 2026). Its 39% volatility also creates a large volatility drag: an 18.6% mean annual return compounds to only 11.9% a year.
- **Top quintiles (Q5):** the highest Sortino (3.4 and 3.3) and strong positive skew (0.8 and 1.3), driven by a few very large months, mostly in 2026.